# 量子谐振子数值计算习题集
# Quantum Harmonic Oscillator Numerical Exercises

这个 notebook 演示了量子谐振子的三个练习：

1. **练习1**: 基态能量计算和误差分析
2. **练习2**: 特征值和特征向量求解
3. **练习3**: 含时量子谐振子演化

---

## 理论背景

### 哈密顿量 / Hamiltonian

一维量子谐振子的哈密顿量为：

$$H = \frac{\hat{p}^2}{2m} + \frac{1}{2}m\omega^2\hat{x}^2$$

在自然单位制 ($\hbar = m = \omega = 1$) 下：

$$H = \frac{\hat{p}^2}{2} + \frac{\hat{x}^2}{2}$$

### 精确解 / Exact Solution

**能级 / Energy Levels:**
$$E_n = \hbar\omega\left(n + \frac{1}{2}\right) = n + \frac{1}{2}, \quad n = 0, 1, 2, ...$$

**基态波函数 / Ground State Wavefunction:**
$$\psi_0(x) = \pi^{-1/4} e^{-x^2/2}$$

**基态能量 / Ground State Energy:**
$$E_0 = \frac{1}{2}$$

In [ ]:
# 导入必要的库 / Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

# 导入我们的量子谐振子模块 / Import our quantum harmonic oscillator module
from quantum_harmonic_oscillator import (
    QuantumHarmonicOscillator,
    TimeDependentHarmonicOscillator
)

# 设置绘图样式 / Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✓ 模块导入成功 / Modules imported successfully")

---

## 练习 1: 基态能量计算和误差分析
## Exercise 1: Ground State Energy Calculation and Error Analysis

### 任务 / Tasks

**a.** 数值计算基态能量期望值 $\langle \psi_0 | H | \psi_0 \rangle$

**b.** 分析误差来源：波函数离散化 vs 积分近似

In [ ]:
# 创建量子谐振子对象 / Create quantum harmonic oscillator object
N = 1000  # 网格点数 / Number of grid points
L = 10.0  # 空间区间半长度 / Half-length of spatial domain

qho = QuantumHarmonicOscillator(N=N, L=L)

print(f"量子谐振子参数 / Quantum Harmonic Oscillator Parameters:")
print(f"  网格点数 / Grid points: N = {N}")
print(f"  空间区间 / Spatial domain: x ∈ [{-L:.1f}, {L:.1f}]")
print(f"  网格间距 / Grid spacing: Δx = {qho.dx:.6f}")
print(f"  ħ = {qho.hbar}, m = {qho.m}, ω = {qho.omega}")

### 1a. 计算基态能量期望值

In [ ]:
# 获取精确基态波函数 / Get exact ground state wavefunction
psi0 = qho.exact_ground_state_wavefunction()

# 绘制基态波函数 / Plot ground state wavefunction
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：波函数 / Left: Wavefunction
ax = axes[0]
ax.plot(qho.x, psi0, 'b-', linewidth=2, label='$\\psi_0(x)$')
ax.fill_between(qho.x, 0, psi0, alpha=0.3)
ax.set_xlabel('Position x', fontsize=12)
ax.set_ylabel('$\\psi_0(x)$', fontsize=12)
ax.set_title('Ground State Wavefunction', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# 右图：概率密度 / Right: Probability density
ax = axes[1]
ax.plot(qho.x, np.abs(psi0)**2, 'r-', linewidth=2, label='$|\\psi_0(x)|^2$')
ax.fill_between(qho.x, 0, np.abs(psi0)**2, alpha=0.3, color='red')

# 添加势能曲线 / Add potential curve
V = 0.5 * qho.x**2
V_normalized = V / V.max() * np.max(np.abs(psi0)**2) * 0.5
ax.plot(qho.x, V_normalized, 'k--', alpha=0.5, label='V(x) (scaled)')

ax.set_xlabel('Position x', fontsize=12)
ax.set_ylabel('$|\\psi_0(x)|^2$', fontsize=12)
ax.set_title('Ground State Probability Density', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 检查归一化 / Check normalization
from scipy.integrate import simpson as simps
norm = simps(np.abs(psi0)**2, qho.x)
print(f"\n归一化检查 / Normalization check: ∫|ψ₀|² dx = {norm:.10f}")

In [ ]:
# 计算能量期望值 / Compute energy expectation value
E0_simpson = qho.compute_energy_expectation(psi0, integration_method='simpson')
E0_trapz = qho.compute_energy_expectation(psi0, integration_method='trapezoid')
E0_exact = qho.exact_energy(0)

print("="*70)
print("基态能量计算结果 / Ground State Energy Results")
print("="*70)
print(f"精确值 / Exact:           E₀ = {E0_exact:.10f}")
print(f"Simpson 方法 / Simpson:   E₀ = {E0_simpson:.10f}  (误差 {abs(E0_simpson - E0_exact):.2e})")
print(f"Trapz 方法 / Trapezoid:   E₀ = {E0_trapz:.10f}  (误差 {abs(E0_trapz - E0_exact):.2e})")
print("="*70)

### 1b. 误差分析

In [ ]:
# 误差分析：改变网格点数 / Error analysis: varying grid points
N_values = [100, 200, 400, 800, 1600, 3200]
results = qho.error_analysis(N_values=N_values)

# 绘制误差分析 / Plot error analysis
qho.plot_error_analysis(results)

### 误差分析结论 / Error Analysis Conclusion

从误差收敛曲线可以看出：

1. **主要误差来源**: 对于较大的 $N$，误差主要来自**波函数离散化**，而不是积分方法
2. **收敛速度**: 误差大致按照 $O(\Delta x^2)$ 收敛
3. **积分方法的影响**: Simpson 和 Trapezoid 方法的差异在大 $N$ 时变得可见，但相对较小

**建议**: 对于精确计算，选择足够大的 $N$ (如 $N \geq 1000$) 比选择高阶积分方法更重要。

---

## 练习 2: 特征值和特征向量求解
## Exercise 2: Eigenvalue and Eigenvector Solver

### 任务 / Tasks

**a.** 对角化哈密顿矩阵，计算前 $k$ 个特征值和特征向量

**b.** 验证数值解的正确性

In [ ]:
# 创建新的量子谐振子对象用于对角化 / Create new QHO object for diagonalization
qho2 = QuantumHarmonicOscillator(N=800, L=10.0)

# 构建哈密顿矩阵 / Build Hamiltonian matrix
import time
start = time.time()
H = qho2.build_hamiltonian_matrix()
build_time = time.time() - start

print(f"哈密顿矩阵构建完成 / Hamiltonian matrix constructed")
print(f"  矩阵大小 / Matrix size: {H.shape}")
print(f"  构建时间 / Build time: {build_time:.4f} s")
print(f"  矩阵是否厄米 / Is Hermitian: {np.allclose(H, H.conj().T)}")

In [ ]:
# 对角化哈密顿矩阵 / Diagonalize Hamiltonian
start = time.time()
eigenvalues, eigenvectors = qho2.diagonalize()
diag_time = time.time() - start

print(f"对角化完成 / Diagonalization completed")
print(f"  对角化时间 / Diagonalization time: {diag_time:.4f} s")
print(f"  计算了 {len(eigenvalues)} 个特征值 / Computed {len(eigenvalues)} eigenvalues")

In [ ]:
# 收敛性测试 / Convergence test
convergence_results = qho2.convergence_test(k_max=10)

In [ ]:
# 绘制能级对比图 / Plot energy level comparison
n_levels = 10
exact_energies = [qho2.exact_energy(n) for n in range(n_levels)]
numerical_energies = eigenvalues[:n_levels]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左图：能级对比 / Left: Energy level comparison
ax = axes[0]
n_array = np.arange(n_levels)
ax.plot(n_array, exact_energies, 'ko-', markersize=10, linewidth=2, label='Exact')
ax.plot(n_array, numerical_energies, 'rx--', markersize=8, linewidth=2, label='Numerical')
ax.set_xlabel('Quantum number n', fontsize=12)
ax.set_ylabel('Energy $E_n$', fontsize=12)
ax.set_title('Energy Levels: Exact vs Numerical', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# 右图：相对误差 / Right: Relative error
ax = axes[1]
rel_errors = [abs(numerical_energies[n] - exact_energies[n]) / exact_energies[n] 
              for n in range(n_levels)]
ax.semilogy(n_array, rel_errors, 'bo-', markersize=8, linewidth=2)
ax.set_xlabel('Quantum number n', fontsize=12)
ax.set_ylabel('Relative Error', fontsize=12)
ax.set_title('Relative Error vs Quantum Number', fontsize=14)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

In [ ]:
# 正交归一性验证 / Orthonormality verification
qho2.verify_orthonormality(k_max=5)

In [ ]:
# 绘制前几个波函数 / Plot first few wavefunctions
qho2.plot_wavefunctions(n_states=6)

---

## 练习 3: 含时量子谐振子
## Exercise 3: Time-Dependent Quantum Harmonic Oscillator

### 问题描述

考虑含时哈密顿量：

$$H(t) = \frac{\hat{p}^2}{2} + \frac{(\hat{q} - q_0(t))^2}{2}$$

其中：
$$q_0(t) = \frac{t}{T}, \quad t \in [0, T]$$

初态为基态 $|\psi_0\rangle = |n=0\rangle$。

### 任务 / Tasks

研究不同 $T$ 值下的时间演化和绝热性。

In [ ]:
# 创建含时量子谐振子对象 / Create time-dependent QHO object
td_qho = TimeDependentHarmonicOscillator(N=400, L=12.0)

# 初态：基态 / Initial state: ground state
psi0 = td_qho.exact_ground_state_wavefunction()

print("含时量子谐振子 / Time-Dependent Quantum Harmonic Oscillator")
print(f"  网格点数 / Grid points: N = {td_qho.N}")
print(f"  空间区间 / Spatial domain: x ∈ [{-td_qho.L:.1f}, {td_qho.L:.1f}]")
print(f"  初态 / Initial state: Ground state |ψ₀⟩")

### 3a. 单个时间演化示例

In [ ]:
# 演化一个中等时间尺度 / Evolve for an intermediate time scale
T = 2.0
Nt = 200

print(f"时间演化参数 / Time evolution parameters:")
print(f"  总时间 / Total time: T = {T}")
print(f"  时间步数 / Time steps: Nt = {Nt}")
print(f"  时间步长 / Time step: dt = {T/Nt:.4f}")
print()

psi_final, psi_history = td_qho.time_evolution_split_step(psi0, T, Nt=Nt, save_interval=10)

In [ ]:
# 绘制时间演化快照 / Plot time evolution snapshots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# 选择几个时刻 / Select several time points
time_indices = [0, len(psi_history)//5, 2*len(psi_history)//5, 
                3*len(psi_history)//5, 4*len(psi_history)//5, -1]

for idx, t_idx in enumerate(time_indices):
    ax = axes[idx]
    t = td_qho.time_history[t_idx]
    psi = psi_history[t_idx]
    
    # 绘制概率密度 / Plot probability density
    ax.plot(td_qho.x, np.abs(psi)**2, 'b-', linewidth=2)
    ax.fill_between(td_qho.x, 0, np.abs(psi)**2, alpha=0.3)
    
    # 标记 q0(t) / Mark q0(t)
    q0_t = t / T
    ax.axvline(x=q0_t, color='r', linestyle='--', alpha=0.7, label=f'$q_0(t)={q0_t:.2f}$')
    
    ax.set_xlabel('Position x', fontsize=11)
    ax.set_ylabel('$|\\psi(x,t)|^2$', fontsize=11)
    ax.set_title(f't = {t:.3f} (t/T = {t/T:.2f})', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-6, 6)

plt.tight_layout()
plt.show()

In [ ]:
# 计算跃迁概率 / Compute transition probabilities
n_max = 10
probs = td_qho.compute_transition_probabilities(psi_final, n_max=n_max)

# 绘制跃迁概率 / Plot transition probabilities
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(range(n_max), probs, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_xlabel('Quantum number n', fontsize=12)
ax.set_ylabel('Transition probability $P_n$', fontsize=12)
ax.set_title(f'Transition Probabilities at T = {T}', fontsize=14)
ax.set_xticks(range(n_max))
ax.grid(True, alpha=0.3, axis='y')

# 添加数值标签 / Add numerical labels
for n in range(n_max):
    if probs[n] > 0.01:
        ax.text(n, probs[n] + 0.01, f'{probs[n]:.3f}', 
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\n跃迁概率汇总 / Transition probability summary (T = {T}):")
print(f"  P₀ (基态) = {probs[0]:.4f}")
print(f"  P₁ = {probs[1]:.4f}")
print(f"  P₂ = {probs[2]:.4f}")
print(f"  激发态总概率 / Total excited prob = {np.sum(probs[1:]):.4f}")

### 3b. 绝热性分析

In [ ]:
# 分析不同 T 值的绝热性 / Analyze adiabaticity for different T values
T_values = [0.2, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0]

print("开始绝热性分析... / Starting adiabaticity analysis...")
print("(这可能需要几分钟 / This may take a few minutes)\n")

adiabatic_results = td_qho.analyze_adiabaticity(T_values, method='split_step')

In [ ]:
# 绘制绝热性分析结果 / Plot adiabaticity analysis results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 基态概率 vs T
ax = axes[0, 0]
ax.semilogx(adiabatic_results['T_values'], adiabatic_results['P0'], 
           'bo-', linewidth=2, markersize=8, label='$P_0$ (ground state)')
ax.axhline(y=0.9, color='r', linestyle='--', alpha=0.5, label='90% threshold')
ax.set_xlabel('Total time T', fontsize=12)
ax.set_ylabel('Ground state probability $P_0$', fontsize=12)
ax.set_title('Adiabaticity: Ground State Population', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 2. 激发态概率 vs T
ax = axes[0, 1]
ax.loglog(adiabatic_results['T_values'], adiabatic_results['P_excited'], 
         'ro-', linewidth=2, markersize=8)
ax.set_xlabel('Total time T', fontsize=12)
ax.set_ylabel('Excited state probability', fontsize=12)
ax.set_title('Excited State Population (log-log)', fontsize=13)
ax.grid(True, alpha=0.3, which='both')

# 3. 平均量子数 vs T
ax = axes[1, 0]
ax.semilogx(adiabatic_results['T_values'], adiabatic_results['mean_n'], 
           'go-', linewidth=2, markersize=8)
ax.set_xlabel('Total time T', fontsize=12)
ax.set_ylabel('Mean quantum number $\\langle n \\rangle$', fontsize=12)
ax.set_title('Mean Excitation Level', fontsize=13)
ax.grid(True, alpha=0.3)

# 4. 最终位置期望值 vs T
ax = axes[1, 1]
ax.semilogx(adiabatic_results['T_values'], adiabatic_results['mean_x_final'], 
           'mo-', linewidth=2, markersize=8, label='$\\langle x(T) \\rangle$')
ax.axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='$q_0(T) = 1$')
ax.set_xlabel('Total time T', fontsize=12)
ax.set_ylabel('Final position $\\langle x(T) \\rangle$', fontsize=12)
ax.set_title('Final Position Expectation Value', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 绝热性结论 / Adiabaticity Conclusions

从上述结果可以看出：

1. **绝热极限** ($T \gg 1$):
   - 基态概率 $P_0 > 0.9$
   - 系统几乎保持在瞬时基态
   - $\langle x(T) \rangle \approx q_0(T) = 1$
   
2. **突然极限** ($T \ll 1$):
   - 显著的能级激发
   - $P_0$ 大幅下降
   - 平均量子数 $\langle n \rangle$ 较大
   
3. **绝热判据**:
   - 当 $T \gtrsim 10$ 时，绝热条件基本满足
   - 绝热参数 $\gamma = T \cdot \Delta E \approx T$ (因为 $\Delta E \approx 1$)
   - 绝热条件：$\gamma \gg 1$ ✓

---

## 总结 / Summary

在这个 notebook 中，我们完成了：

### 练习 1: 基态能量计算
- ✓ 数值计算基态能量期望值
- ✓ 系统地分析了误差来源
- ✓ 验证了误差收敛性

**关键结论**: 波函数离散化是主要误差来源

### 练习 2: 特征值求解
- ✓ 成功对角化哈密顿矩阵
- ✓ 验证了数值解的正确性
- ✓ 检查了正交归一性

**关键结论**: 有限差分方法配合 LAPACK 对角化提供了准确且高效的解

### 练习 3: 含时演化
- ✓ 实现了时间演化算法
- ✓ 研究了绝热性
- ✓ 分析了跃迁概率

**关键结论**: 绝热定理的数值验证，$T \gtrsim 10$ 时系统保持在基态

---

## 扩展练习 / Extended Exercises

1. **不同势能**: 修改代码以处理双井势、Morse 势等
2. **更高维**: 扩展到二维或三维谐振子
3. **多粒子系统**: 研究两个耦合谐振子
4. **其他时间演化方法**: 比较 Crank-Nicolson、Magnus 展开等
5. **量子相干性**: 计算纠缠熵、保真度等

---

## 参考文献 / References

1. Griffiths, D. J., & Schroeter, D. F. (2018). *Introduction to Quantum Mechanics* (3rd ed.). Cambridge University Press.
2. Sakurai, J. J., & Napolitano, J. (2017). *Modern Quantum Mechanics* (2nd ed.). Cambridge University Press.
3. Press, W. H., et al. (2007). *Numerical Recipes: The Art of Scientific Computing* (3rd ed.). Cambridge University Press.
4. Landau, R. H., Páez, M. J., & Bordeianu, C. C. (2015). *Computational Physics: Problem Solving with Python* (3rd ed.). Wiley-VCH.